# Commande RST d'un double intégrateur


Le but de ce notebook est de calculer et de simuler un correcteur RST agissant sur un modèle d'une balle sur poutre.

In [ ]:
from Models.BallBeam import ballbeam_config
from Models.BallBeam.StateSpace import LinearStateSpaceModel
from Models.BallBeam.TransferFunctions import TransferFunctionModel

from Simulation.simulation import TFSimulator
from Simulation.simulation import HybridSim

from Simulation.runners import *

from Metrics_Plotting.SimLog import SimLog
from Metrics_Plotting.Plotting import Plotting
from Metrics_Plotting.Metrics import Metrics

from Control.DiscretePID import DiscretePID
from Control.RSTController import RSTController

from Utils import computeRST
from Utils import utils

import numpy as np
import control as ct
import matplotlib.pyplot as plt

%matplotlib inline

Configuration de la simulation:


T est le temps total de simulation et dt est le temps d'échantillonage du correcteur

In [2]:
ballbeam_config.T=6
ballbeam_config.dt=0.05

In [ ]:
X_0 = np.array([[0],[0]])                              # état initial
double_int = TransferFunctionModel(ballbeam_config)    # instance de TransferFunctionModel
Double_int_sim = TFSimulator(double_int.Tf_dis, X_0)   # simulateur de fonctions de transfert discrètes
A = double_int.Tf_dis.den_list[0][0]                   # extraction du dénominateur A(z)
B = double_int.Tf_dis.num_list[0][0]                   # extraction du numérateur B(z)

In [4]:
### affichage 
print(double_int.Tf_cont)
print(double_int.Tf_dis)

<TransferFunction>: sys[0]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

  0.21
  ----
  s^2
<TransferFunction>: sys[0]$sampled
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = 0.05

  0.0002625 z + 0.0002625
  -----------------------
       z^2 - 2 z + 1


Fonction de transfert désirée 

Ici, on veut que le modèle agisse comme un second ordre.

 On commence par spécifier son dénominateur en dans le domaine continu, puis on discrétise les poles, $z_p=e^{s_pT_s}$

In [5]:
omega0=2.2
omega_carr=omega0*omega0
zeta=0.7

Am_s=[(1/(omega_carr)),((2.0*zeta)/omega0),1.0]
Am_d=np.poly(np.exp(np.roots(Am_s)*ballbeam_config.dt))

Bm=sum(Am_d) ## méthode pour assurer un gain unitaire constant Am(1)

Tf_des=ct.tf(Bm,Am_d,ballbeam_config.dt)


In [6]:
### affichage
print(Tf_des)
print("s poles:", np.roots(Am_s))
print("z poles:", np.roots(Am_d))
print("radius:", np.abs(np.roots(Am_d)))
print(Tf_des.dcgain())

<TransferFunction>: sys[2]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = 0.05

          0.0112
  ----------------------
  z^2 - 1.846 z + 0.8573
s poles: [-1.54+1.57111425j -1.54-1.57111425j]
z poles: [0.92303449+0.07265915j 0.92303449-0.07265915j]
radius: [0.92588985 0.92588985]
1.0


In [7]:
## réponse inditielle de la fonction de transfert calculée par la librairie
t = np.arange( 0,ballbeam_config.T,ballbeam_config.dt)
_, y_lib = ct.step_response(Tf_des, t)


In [ ]:
## polynome de l'observateur, ne change pas la réponse iditielle mais modifie celle du rejet des perturbations
A0=[1,-0.9]

Synthèse du correcteur RST:
Pour inclure le polynome A0 il faut l'inclure dans la méthode en faisant np.polymul(Am_d,A0), True/False détermine la présence ou non de l'effet intégral $(1-z^-1)S'=S$.

In [ ]:
S, R, T, H_cl = computeRST.Compute_Denominator_Matching_RST(Am_d, double_int.Tf_dis, True, A0)

Plusieurs données sont affichées. Les plus importantes sont les polynômes R, S et T, la fonction de transfert reconstruite et le gain en régime permanent (Closed-loop DC gain).

### Marges de stabilité

La fonction de transfert en boucle ouverte est $L(z) = \frac{B(z)\,R(z)}{A(z)\,S(z)}$.

In [ ]:
L_rst = double_int.Tf_dis * ct.tf(np.squeeze(R.num[0][0]), np.squeeze(S.num[0][0]), ballbeam_config.dt)
metrics = Metrics()
metrics.Stability(L_rst)

### Simulation
On crée une instance du controleur RST initialisé avec les polynomes R,S,T et on utilise le simulateur discret. Le premier True/False détermine la présence d'une perturbation constante au bout de la troisième seconde. Le deuxième active (si True) la saturation de l'actionneur.

In [ ]:
rst_controller = RSTController(R, S, T)
logger_rst = SimLog()

reference = 1.0
y_0 = 0.0

logger_rst = run_discrete_control(Double_int_sim, rst_controller, ballbeam_config, reference, y_0, logger_rst, True, True)

### Métriques de performance

Dépassement, temps de montée (10 %→90 %) et temps d'établissement (bande ±10 %) calculés sur la phase de suivi de consigne (avant perturbation).

In [ ]:
metrics.response_data(logger_rst, reference)

Création de la fonction de transfert perturbation/sortie $Gd=\frac{BS}{AS+BR}$ et simulation de sa réponse inditielle

In [ ]:
A = np.squeeze(A)
B = np.squeeze(B)
R_arr = np.squeeze(R.num[0][0])
S_arr = np.squeeze(S.num[0][0])

num = np.polymul(B, S_arr)
den = np.polyadd(np.polymul(A, S_arr), np.polymul(B, R_arr))

Gd = ct.tf(num, den, ballbeam_config.dt)
print(Gd)
print("DC gain Gd:", ct.dcgain(Gd))

t = np.asarray(t).flatten()
_, d_resp = ct.step_response(Gd, t)
d_resp = np.squeeze(d_resp)

Affichage des trois réponses

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax[0].step(logger_rst.t_hist, logger_rst.y_hist, label='y(t)')
ax[0].step(t, np.squeeze(y_lib), linestyle='--', color='gray', label='Modèle désiré')
ax[0].axhline(y=reference, color='green', linestyle='--', linewidth=1, label=f'Consigne r={reference}')
ax[0].axvline(x=3, color='orange', linestyle='--', linewidth=1, label='Perturbation')
ax[0].set_title("Sortie y(t)")
ax[0].set_ylabel("Position [m]")
ax[0].legend()
ax[0].grid()

ax[1].step(logger_rst.t_hist, logger_rst.u_hist, label='u(t)')
ax[1].axhline(y= 10, color='red', linestyle=':', linewidth=1, label='Saturation ±10')
ax[1].axhline(y=-10, color='red', linestyle=':', linewidth=1)
ax[1].axvline(x=3, color='orange', linestyle='--', linewidth=1, label='Perturbation')
ax[1].set_title("Commande u(t)")
ax[1].set_ylabel("u [V]")
ax[1].legend()
ax[1].grid()

ax[2].step(t, d_resp, label='$G_d$ — perturbation unitaire')
ax[2].axvline(x=3, color='orange', linestyle='--', linewidth=1)
ax[2].set_title("Réponse perturbation→sortie $G_d(z)$")
ax[2].set_ylabel("Δy [m]")
ax[2].set_xlabel("Temps [s]")
ax[2].legend()
ax[2].grid()

plt.tight_layout()
plt.show()